In [3]:
import pandas as pd, plotly.express as px
sector  = pd.read_csv("../data/processed/monthly_by_sector.csv")
country = pd.read_csv("../data/processed/country_summary.csv")
group   = pd.read_csv("../data/processed/monthly_by_group.csv")
pay_yr  = pd.read_csv("../data/manual/payments_annual.csv")
pay_q   = pd.read_csv("../data/manual/payments_quarterly.csv")
sector = sector[sector["month"] < "2026-10"]


In [4]:
#Is the number of claimed victims rising?

px.line(sector.groupby("month")["victims"].sum().reset_index(), x="month", y="victims")

#Change from first full year to last full year, 

In [5]:
#Which Top 3 sectors are hit most?

sector.groupby("sector")["victims"].sum().sort_values(ascending=False).head(3)

sector
Professional Services    5198
Manufacturing            4653
Technology               3476
Name: victims, dtype: int64

In [6]:
#Which countries are hit most, raw and for their size?

country.sort_values("victims_12m", ascending=False).head(10)

,country,victims_12m,gdp_usd,victims_per_100bn_gdp
141,US,4051,2.929801e+13,13.826876
144,Unknown,618,NaN,NaN
35,DE,479,4.685593e+12,10.222827
48,GB,405,3.695540e+12,10.959158
24,CA,356,2.270076e+12,15.682293
65,IT,280,2.383436e+12,11.747748
46,FR,271,3.160443e+12,8.574748
19,BR,229,2.185822e+12,10.476610
42,ES,212,1.725672e+12,12.285072
61,IN,204,3.760813e+12,5.424358


In [7]:
#How the two top-10 lists differ

country[country["victims_12m"] > 20].sort_values("victims_per_100bn_gdp", ascending=False).head(10)

,country,victims_12m,gdp_usd,victims_per_100bn_gdp
131,TH,114,5.293855e+11,21.534401
95,MY,81,4.222270e+11,19.183993
7,AR,108,6.383655e+11,16.918209
24,CA,356,2.270076e+12,15.682293
31,CO,63,4.205040e+11,14.982020
106,PE,43,2.917515e+11,14.738569
34,CZ,51,3.470826e+11,14.693910
1,AE,81,5.523249e+11,14.665281
123,SG,84,5.728773e+11,14.662827
141,US,4051,2.929801e+13,13.826876


In [59]:
#Which groups drive the activity?

group.groupby("group")["victims"].sum().sort_values(ascending=False).head(10)

#Top groups' share, and whether it's concentrated in a few

group
qilin           2336
akira           1632
lockbit3        1598
play            1288
clop            1174
incransom        968
thegentlemen     922
ransomhub        842
dragonforce      658
safepay          591
Name: victims, dtype: int64

In [50]:
#Are victims rising while payments fall?


# 1. Victims per year (from the monthly sector table)

victims_yr = (
    sector.assign(year=sector["month"].astype(str).str[:4].astype(int))
          .groupby("year", as_index=False)["victims"].sum()
)

# 2. Line it up with payments

pay_yr["year"] = pay_yr["year"].astype(int)   # avoids int-vs-float merge misses
econ = victims_yr.merge(pay_yr[["year", "total_paid_usd"]], on="year", how="inner")

print(econ.round(1))

   year  victims  total_paid_usd
0  2023     4904      1230000000
1  2024     5979       892000000
2  2025     8136       820000000


In [ ]:
#Claimed victims rose about 36% from 2024 to 2025 while on-chain payments fell about 8%; I concur with the moderate confidence analysis by Chainanalysis that ransomware is becoming a high-volume, lower-yield business.